# Word Embeddings and Emotion Recognition
## Project: GoEmotions Multi-class/Multi-label Classification

This notebook demonstrates the full end-to-end workflow for:
1. Training custom **Word2Vec** models.
2. Analyzing and Visualizing embeddings (t-SNE).
3. Loading pretrained **GloVe** vectors.
4. Training Neural Networks (**Dense, CNN, BiLSTM**) for emotion classification.
5. Comparing performance against TF-IDF Baselines.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.append(os.path.abspath('../src'))

from utils import setup_logging, clean_text, save_plot
from word2vec_training import Word2VecTrainer
from embedding_analysis import EmbeddingAnalyzer
from glove_loader import GloveLoader
from emotion_classifier import EmotionClassifier
from baseline_models import BaselineModels

logger = setup_logging()
print("Environment Setup Complete.")

## 1. Word2Vec Training
We train a custom Word2Vec model on a domain corpus.

In [ ]:
# Training with multiple hyperparameters
trainer = Word2VecTrainer()
trainer.load_corpus_from_file('../data/corpus/news_sample.txt')
trainer.train_variants(output_dir='../models/word2vec')

# Evaluate one model
model_files = os.listdir('../models/word2vec')
if model_files:
    sample_path = os.path.join('../models/word2vec', model_files[0])
    trainer.evaluate_model(sample_path)

## 2. Embedding Visualization (t-SNE)
Visualizing the semantic clusters in 2D and 3D.

In [ ]:
analyzer = EmbeddingAnalyzer()
if model_files:
    analyzer.load_model(sample_path)
    words, vectors = analyzer.extract_top_words(top_n=500)
    analyzer.plot_tsne(words, vectors, filename_prefix='w2v_final')
    analyzer.plot_tsne_3d(words, vectors, filename_prefix='w2v_final')

## 3. Data Loading & Preprocessing (GoEmotions)
Analysis of class imbalance and distribution.

In [ ]:
df = pd.read_csv('../go_emotions_dataset.csv')
emotion_cols = df.columns[3:] # Columns after id, text, example_very_unclear

# Visualize distribution
plt.figure(figsize=(15, 6))
df[emotion_cols].sum().sort_values(ascending=False).plot(kind='bar')
plt.title("Emotion Label Distribution")
plt.show()

# Single label target for this demo (most frequent emotion excluding neutral if possible)
# Or we can treat it as multi-label as per requirements.
labels = df[emotion_cols].values
texts = df['text'].values

classifier = EmotionClassifier(max_words=10000, max_len=50)
# Using multi-label support for GoEmotions
X_train, X_val, X_test, y_train, y_val, y_test = classifier.prepare_data(texts, labels, multi_label=True)

## 4. Embedding Matrix (GloVe)
Mapping GloVe vectors to our tokenizer vocabulary.

In [ ]:
loader = GloveLoader(glove_path='../embeddings/glove.6B.100d.txt')
loader.load_embeddings()
loader.calculate_coverage(classifier.tokenizer.word_index.keys())
classifier.embedding_matrix = loader.create_embedding_matrix(classifier.tokenizer.word_index)
print("Embedding Matrix Prepared.")

## 5. Neural Model Training
Training BiLSTM and CNN architectures.

In [ ]:
num_classes = len(emotion_cols)

# Build BiLSTM
bilstm_model = classifier.build_bilstm_model(num_classes, multi_label=True)
history = classifier.train_model(bilstm_model, X_train, y_train, X_val, y_val, epochs=5, multi_label=True)

classifier.plot_history(history, model_name='BiLSTM')
classifier.evaluate(bilstm_model, X_test, y_test, label_names=emotion_cols, multi_label=True)

## 6. Baseline Comparisons
Comparing Neural models with Logistic Regression and Random Forest.

In [ ]:
# For baselines, we use a single label (dominant emotion)
y_train_single = np.argmax(y_train, axis=1)
y_test_single = np.argmax(y_test, axis=1)

baselines = BaselineModels()
X_train_text = [" ".join([classifier.tokenizer.index_word.get(idx, "") for idx in seq]) for seq in X_train]
X_test_text = [" ".join([classifier.tokenizer.index_word.get(idx, "") for idx in seq]) for seq in X_test]

baselines.train_and_evaluate(X_train_text, y_train_single, X_test_text, y_test_single)
baselines.compare_performances()